In [0]:
import requests
from html.parser import HTMLParser
from urllib.parse import urljoin
from datetime import timezone , datetime
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import yaml 
import re
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType 

In [0]:
with open("../configs/config.yaml", "r") as file:
    config = yaml.safe_load(file)

bls_config = config["sources"]["bls"]
print(bls_config)

BLS_URL = bls_config["url"]

HEADERS = {
    "User-Agent": bls_config["user_agent"]
}

BLS_TARGET_PATH = bls_config["target_path"] 

population_config = config["sources"]["population"]

POPULATION_TARGET_PATH = population_config["target_path"]

POPULATION_URL = population_config["url"]

RAW_VOLUMN_PATH = config["volume"]["raw_path"]


In [0]:
## Create Raw Folders if not present

dbutils.fs.mkdirs(BLS_TARGET_PATH)
dbutils.fs.mkdirs(POPULATION_TARGET_PATH)


## Verify Volumn Structure
files = dbutils.fs.ls(RAW_VOLUMN_PATH)

for i in files:
    print(i.name)

In [0]:
## Check BLS Connections & Response

response = requests.get(
    BLS_URL,
    headers=HEADERS,
    timeout=30
)

print(response.status_code)

In [0]:
class BLSParser(HTMLParser):

    def __init__(self):
        super().__init__()
        self.files = []

    def handle_starttag(self, tag, attrs):
        if tag == "a":
            for key, value in attrs:
                if key == "href":
                    file_name = Path(value).name
                    if file_name.startswith("pr."):
                        self.files.append(file_name)

In [0]:
parser = BLSParser()

parser.feed(response.text)

bls_files = parser.files

print("Count:", len(bls_files))

for file in bls_files:
    print(file)

In [0]:
# Parse file metadata from BLS directory listing

pattern = re.compile(
    r'(?P<modified>\d{1,2}/\d{1,2}/\d{4}\s+\d{1,2}:\d{2}\s+[AP]M)'
    r'\s+'
    r'(?P<size>\d+)'
    r'\s+'
    r'<A HREF="(?P<href>[^"]+)">(?P<file_name>pr\.[^<]+)</A>',
    re.IGNORECASE
)

bls_inventory = []

for match in pattern.finditer(response.text):

    file_name = match.group("file_name")

    source_modified_time = datetime.strptime(
        match.group("modified"),
        "%m/%d/%Y %I:%M %p"
    )

    file_size = int(match.group("size"))

    source_url = urljoin(
        BLS_URL,
        match.group("href")
    )

    bls_inventory.append({
        "file_name": file_name,
        "source_url": source_url,
        "file_size": file_size,
        "source_modified_time": source_modified_time
    })

In [0]:
print("Files discovered:", len(bls_inventory))
inventory_df = spark.createDataFrame(bls_inventory)

# for file in bls_inventory:
#     print(file)

In [0]:
# Fully qualified manifest table name

catalog_name = config["catalog"]["name"]
bronze_schema = config["schemas"]["bronze"]
manifest_table = config["tables"]["ingestion_manifest"]

MANIFEST_TABLE = f"{catalog_name}.{bronze_schema}.{manifest_table}"

print(MANIFEST_TABLE)

manifest_schema = StructType([
    StructField("source", StringType(), False),
    StructField("file_name", StringType(), False),
    StructField("file_path", StringType(), False),
    StructField("file_size", LongType(), True),
    StructField("source_modified_time", TimestampType(), True),
    StructField("ingestion_time", TimestampType(), False),
    StructField("status", StringType(), False)
])

In [0]:
manifest_df = spark.table(MANIFEST_TABLE)


bls_manifest_df = (
    manifest_df
    .filter(
        (F.col("source") == "BLS") &
        (F.col("status") == "SUCCESS")
    )
)


window_spec = (
    Window
    .partitionBy("source", "file_name")
    .orderBy(
        F.col("source_modified_time").desc(),
        F.col("ingestion_time").desc()
        )
)

latest_manifest_df = (
    bls_manifest_df
    .withColumn(
        "rn",
        F.row_number().over(window_spec)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

display(latest_manifest_df)

In [0]:
comparison_df = (
    inventory_df.alias("src")
    .join(
        latest_manifest_df.alias("m"),
        F.col("src.file_name") == F.col("m.file_name"),
        "left"
    )
    .select(
        F.col("src.file_name"),
        F.col("src.source_url"),
        F.col("src.file_size").alias("source_file_size"),
        F.col("src.source_modified_time"),

        F.col("m.file_name").alias("manifest_file_name"),
        F.col("m.file_size").alias("manifest_file_size"),
        F.col("m.source_modified_time").alias("manifest_modified_time")
    )
    .withColumn(
        "action",
        F.when(
            F.col("manifest_file_name").isNull(),
            "NEW"
        )
        .when(
            (F.col("source_file_size") != F.col("manifest_file_size")) |
            (F.col("source_modified_time") != F.col("manifest_modified_time")),
            "CHANGED"
        )
        .otherwise("UNCHANGED")
    )
)

display(comparison_df)

In [0]:
files_to_download = (
    comparison_df
    .filter(F.col("action").isin("NEW","CHANGED"))
    .collect()
)
print(files_to_download)

In [0]:
download_results = []

for file in files_to_download:
    file_name = file["file_name"]
    source_url = file["source_url"]
    file_size = file["source_file_size"]
    source_modified_time = file["source_modified_time"]

    target_file_path = f"{BLS_TARGET_PATH}/{file_name}"

    try:
        file_response = requests.get(
            source_url,
            headers=HEADERS,
            timeout=30
        )
    
        file_response.raise_for_status()

        with open(target_file_path,"wb") as file:
            file.write(file_response.content)

        download_results.append({
            "source": "BLS",
            "file_name": file_name,
            "file_path": target_file_path,
            "file_size": file_size,
            "source_modified_time": source_modified_time,
            "ingestion_time": datetime.now(timezone.utc).replace(tzinfo=None),
            "status": "SUCCESS"
        })

        print(f"Downloaded: , {file_name}")

    except Exception as e:

        download_results.append({
            "source": "BLS",
            "file_name": file_name,
            "file_path": target_file_path,
            "file_size": file_size,
            "source_modified_time": source_modified_time,
            "ingestion_time": datetime.now(timezone.utc).replace(tzinfo=None),
            "status": "FAILED"
        })

        print(f"Failed :, {file_name}")
        

In [0]:
for file in dbutils.fs.ls(BLS_TARGET_PATH):
    print(file.name, file.size)

In [0]:
if download_results:

    try:
        download_manifest_df = spark.createDataFrame(
            download_results,
            schema=manifest_schema
        )

        download_manifest_df.write \
            .mode("append") \
            .saveAsTable(MANIFEST_TABLE)

        print("BLS manifest updated successfully")

    except Exception as e:
        print(f"BLS manifest write failed: {e}")
        raise

In [0]:
removed_results = []

removed_files_df = (
    latest_manifest_df.alias("m")
    .join(
        inventory_df.alias("src"),
        F.col("m.file_name") == F.col("src.file_name"),
        "left_anti"
    )
)

display(removed_files_df)

if removed_files_df.count() > 0:
    for file in removed_files_df.collect():
        removed_results.append({
        "suorce": "BLS",
        "file_name": file["file_name"],
        "file_path": file["file_path"],
        "file_size": file["file_size"],
        "source_modified_time": file["source_modified_time"],
        "ingestion_time": datetime.now(timezone.utc).replace(),
        "status": "REMOVED"
        })


if removed_results:
    removed_manifest = spark.createDataFrame(
        removed_results,
        schema= manifest_schema
    )

    removed_manifest.write \
        .mode("append") \
            .saveAsTable(MANIFEST_TABLE)
            


In [0]:
## Population API config

population_config = config["sources"]["population"]

POPULATION_URL = population_config["url"]
POPULATION_TARGET_PATH = population_config["target_path"]
POPULATION_FILE_NAME = population_config["file"]


In [0]:
## Call Population API

population_response = requests.get(
    POPULATION_URL,
    timeout=30
)

population_response.raise_for_status()

print("status code : ", population_response.status_code)

population_payload= population_response.json()
print(population_payload.keys())



In [0]:
required_keys = {"columns", "data"}

missing_keys = required_keys - population_payload.keys()

if missing_keys:
    raise ValueError(
        f"population api response is missing required keys: {missing_keys}"
        )
    
print("Population API response validation passed")


In [0]:
population_file_path = (
    f"{POPULATION_TARGET_PATH}/{POPULATION_FILE_NAME}"
)
population_content = population_response.content


existing_population_content = None

try:
    with open(population_file_path,"rb") as file:
        existing_population_content = file.read()

except FileNotFoundError:
    pass



In [0]:
if existing_population_content is None:

    population_action = "NEW"

elif existing_population_content != population_content:

    population_action = "CHANGED"

else:

    population_action = "UNCHANGED"


print("Population action:", population_action)

In [0]:
population_results = []

if population_action in ("NEW", "CHANGED"):

    try:
        with open(population_file_path, "wb") as file:
            file.write(population_content)

        population_results.append({
            "source": "POPULATION",
            "file_name": POPULATION_FILE_NAME,
            "file_path": population_file_path,
            "file_size": len(population_content),
            "source_modified_time": None,
            "ingestion_time": datetime.now(timezone.utc).replace(tzinfo=None),
            "status": "SUCCESS"
        })

        print(f"{population_action}: population.json saved successfully")

    except Exception as e:

        population_results.append({
            "source": "POPULATION",
            "file_name": POPULATION_FILE_NAME,
            "file_path": population_file_path,
            "file_size": len(population_content),
            "source_modified_time": None,
            "ingestion_time": datetime.now(timezone.utc).replace(tzinfo=None),
            "status": "FAILED"
        })

        print(f"Population ingestion failed: {e}")

else:
    print("Population response unchanged. Skipping write.")

In [0]:
if population_results:

    try:
        population_manifest_df = spark.createDataFrame(
            population_results,
            schema=manifest_schema
        )

        population_manifest_df.write \
            .mode("append") \
            .saveAsTable(MANIFEST_TABLE)

        print("Population manifest updated successfully")

    except Exception as e:
        print(f"Population manifest write failed: {e}")
        raise

In [0]:
manifest_df = spark.table(MANIFEST_TABLE)
display(manifest_df)

In [0]:
dbutils.fs.rm(population_file_path)

In [0]:
# series_path = (
#     "/Volumes/adb_rearc_assignment_workspace/"
#     "bronze/raw_source_files/bls/pr.series"
# )

# series_df = (
#     spark.read
#     .option("header", "true")
#     .option("sep", "\t")
#     .csv(series_path)
# )

# display(series_df.limit(10))

# series_df.printSchema()

In [0]:
all_data_df = spark.table(
    "adb_rearc_assignment_workspace.bronze.bronze_bls_all_data"
)

display(all_data_df.limit(10))

all_data_df.printSchema()

In [0]:
population_df = spark.table(
    "adb_rearc_assignment_workspace.bronze.bronze_population"
)

display(population_df.limit(10))
population_df.printSchema() 

In [0]:
display(
    spark.table(
        "adb_rearc_assignment_workspace.bronze.bronze_bls_period"
    )
)

In [0]:
measure_df = spark.table(
    "adb_rearc_assignment_workspace.bronze.bronze_bls_measure"
)

display(measure_df.limit(10))
measure_df.printSchema()

In [0]:
sector_df = spark.table(
    "adb_rearc_assignment_workspace.bronze.bronze_bls_sector"
)

display(sector_df.limit(10))
sector_df.printSchema()

In [0]:
class_df = spark.table(
    "adb_rearc_assignment_workspace.bronze.bronze_bls_class"
)

display(class_df.limit(10))
class_df.printSchema()


In [0]:
duration_df = spark.table(
    "adb_rearc_assignment_workspace.bronze.bronze_bls_duration"
)

display(duration_df.limit(10))
duration_df.printSchema()

In [0]:
seasonal_df = spark.table(
    "adb_rearc_assignment_workspace.bronze.bronze_bls_seasonal"
)

display(seasonal_df.limit(10))
seasonal_df.printSchema()

In [0]:
footnote_df = spark.table(
    "adb_rearc_assignment_workspace.bronze.bronze_bls_footnote"
)

display(footnote_df.limit(10))
footnote_df.printSchema()

In [0]:
display(
    spark.table(
        "adb_rearc_assignment_workspace.gold.gold_q1_population_stats_2013_2018"
    )
)

In [0]:
display(
    spark.table(
        "adb_rearc_assignment_workspace.gold.gold_q2_bls_best_year_by_series"
    )
    .orderBy("series_id", "best_year")
    .limit(20)
)

In [0]:
display(
    spark.table(
        "adb_rearc_assignment_workspace.gold.gold_q3_bls_q01_with_population"
    )
    .orderBy("year")
)

In [0]:
from pyspark.sql import functions as F

q2_df = spark.table(
    "adb_rearc_assignment_workspace.gold.gold_q2_bls_best_year_by_series"
)

q2_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("series_id").alias("distinct_series")
).show()

In [0]:
(
    q2_df
    .groupBy("series_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy("series_id")
    .show(100, truncate=False)
)

In [0]:
(
    q2_df
    .filter(
        F.col("sector_name").isNull() |
        F.col("class_text").isNull() |
        F.col("measure_text").isNull() |
        F.col("duration_text").isNull() |
        F.col("seasonal_text").isNull()
    )
    .show(100, truncate=False)
)